In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Create a folder in the root directory
!mkdir -p "/content/drive/My Drive/nlptest"
%cd "/content/drive/My Drive/nlptest"

# load the dataset using pandas
import pandas as pd
df = pd.read_csv("bbc_text_cls.csv")

In [ ]:
# Some basic info on the dataset
print(df.describe())
print(df.info())
print(df.shape)
print(df.head(3))

In [ ]:
# extract the contents of the text column from the df dataframe to a list
corpus = df['text'].tolist()
print("Number of items in corpus:", len(corpus))
print("Item 1: ")
for item in corpus[:1]:
  print(item)

In [ ]:
# Now let's calculate the TF-IDF matrix
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=30,max_df=0.6)
X = vectorizer.fit_transform(corpus)
vectorizer.get_feature_names_out()

print(f"n_samples: {X.shape[0]}, n_features: {X.shape[1]}")

print(("Proportion of empty cells"))
import numpy as np
print(f"{X.nnz / np.prod(X.shape):.3f}")

In [ ]:
from collections import defaultdict
from time import time
from sklearn import metrics
from sklearn.cluster import KMeans

# The code in the block below is adapted from the sklearn tutorial on
# k-means clustering for document.
# https://scikit-learn.org/1.5/auto_examples/text/plot_document_clustering.html

# lists to allow us to collect information on performance.
evaluations = []
evaluations_std = []

# direct access variable to the 'gold standard / correct' labels.
# Remember. These are not being used for the clustering. We are just using these
# to try to evaluate how good our clustering is.
labels = df['labels']

# Scikit learn tutorial derived function to cluster and then evaluate the cluster
def fit_and_evaluate(km, X, name=None, n_runs=5):
    """ km = the k means classifier instance
        X = The data
        n_runs = how many times the clustering algorithm is run. """
    # determine the name
    name = km.__class__.__name__ if name is None else name

    # variable to collect the training times so that we can display the average
    train_times = []
    # variable to save the scores. Keep in mind we are saving lots of different
    # score types.
    scores = defaultdict(list)
    # For each run
    for seed in range(n_runs):
        # set a random seed based on the runs index
        km.set_params(random_state=seed)
        # save time 'now' so that we can calculate the time taken to fit.
        t0 = time()
        # fit the k means model on the data X
        km.fit(X)
        # get the time 'now' and compare that to t0 above to calc the time taken
        train_times.append(time() - t0)
        # use the scikit metrics package to calculate various scores - see slides
        # append each of these results into the 'scores' container.
        scores["Homogeneity"].append(metrics.homogeneity_score(labels, km.labels_))
        scores["Completeness"].append(metrics.completeness_score(labels, km.labels_))
        scores["Silhouette Coefficient"].append(
            metrics.silhouette_score(X, km.labels_, sample_size=2000)
        )
    # restructure the training times into a numpy array and print.
    train_times = np.asarray(train_times)
    print(f"clustering done in {train_times.mean():.2f} ± {train_times.std():.2f} s ")
    # calculate the mean and standard deviation of the training times.
    evaluation = {
        "estimator": name,
        "train_time": train_times.mean(),
    }
    evaluation_std = {
        "estimator": name,
        "train_time": train_times.std(),
    }
    # iterate over each score type in the scores container
    for score_name, score_values in scores.items():
        # calculate the mean and standard deviation of the current score type then print
        mean_score, std_score = np.mean(score_values), np.std(score_values)
        print(f"{score_name}: {mean_score:.3f} ± {std_score:.3f}")
        # save the score type and variable
        evaluation[score_name] = mean_score
        evaluation_std[score_name] = std_score
    evaluations.append(evaluation)
    evaluations_std.append(evaluation_std)

In [ ]:
# Construct the k means clustering object that expects 10 clusters and pass it to our fit function
kmeans = KMeans(
    n_clusters=10,
    max_iter=100,
    n_init=10,
)
print(f"Trying 10 clusters")
fit_and_evaluate(kmeans, X, name="KMeans\tf-idf vectors")
print("")

# Try again with different cluser sizes
for n_clusters in [2, 3, 4, 5, 6]:
    kmeans = KMeans(
        n_clusters=n_clusters,
        max_iter=1000,
        n_init=100,
    )
    print(f"Trying {n_clusters} clusters")
    fit_and_evaluate(kmeans, X, name="KMeans\tf-idf vectors")
    print("")

In [ ]:
# Construct the k means clustering object that expects 10 clusters and pass it to our fit function
kmeans = KMeans(
    n_clusters=5,
    max_iter=1000,
    n_init=1000,
)
print(f"Trying 10 clusters")
fit_and_evaluate(kmeans, X, name="KMeans\tf-idf vectors")
print("")